[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/bgplvm_oil100.ipynb)

# Bayesian GPLVM on oil (100) with `gpc`

Collapsed / psi Bayesian GPLVM ([CIP-0017](https://github.com/SheffieldML/GPc/blob/main/cip/cip0017.md)); native twin [bgplvm_oil100](https://github.com/SheffieldML/GPc/blob/main/docs/tutorials/bgplvm_oil100.md).

Requires `gpc>=0.1.0a10` (also on `0.1.0a7`).

```python
m = gpc.models.BayesianGPLVM(Y, input_dim=2, num_inducing=10)  # RBF-ARD default
m.optimize()
mu_x, s_x = m.latent_mean(), m.latent_variance()
```

**Notes**

* Default approx is VarDTC (`approx="vardtc"`); `elbo() == -objective()`.
* SVI Bayesian GPLVM remains CLI-first (`./bgplvm --inference svi`) — not this notebook.


In [ ]:
import importlib
import re
import subprocess
import sys


def _gpc_ver_tuple(v: str):
    """Parse ``0.1.0a10`` / ``0.1.0`` into a comparable tuple (final > any alpha)."""
    m = re.fullmatch(r"(\d+)\.(\d+)\.(\d+)(?:a(\d+))?", str(v))
    if not m:
        return (0, 0, 0, 0)
    maj, mir, pat, alpha = m.groups()
    pre = int(alpha) if alpha is not None else 10**9
    return (int(maj), int(mir), int(pat), pre)


_MIN = "0.1.0a10"
_need = _gpc_ver_tuple(_MIN)


def _ensure_gpc():
    try:
        import gpc as _gpc
        if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) >= _need:
            return _gpc
    except ImportError:
        pass
    # Prefer TestPyPI for pre-release alphas; real PyPI for dependencies (CIP-002C).
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--no-cache-dir",
            "--index-url",
            "https://test.pypi.org/simple/",
            "--extra-index-url",
            "https://pypi.org/simple/",
            f"gpc>={_MIN}",
        ]
    )
    if "gpc" in sys.modules:
        importlib.reload(sys.modules["gpc"])
    import gpc as _gpc
    if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) < _need:
        raise RuntimeError(
            f"need gpc>={_MIN}, got {getattr(_gpc, '__version__', '?')}"
        )
    return _gpc


gpc = _ensure_gpc()


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc
from matplotlib import pyplot as plt

try:
    import pods
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pods"])
    import pods

try:
    import matplotlib
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])


## Load oil_100

In [ ]:
data = pods.datasets.oil_100()
Y = np.asarray(data["X"], dtype=float)
labels = np.asarray(data["Y"], dtype=float)
cls = np.argmax(labels, axis=1)
print(Y.shape)

## Fit collapsed Bayesian GPLVM

In [ ]:
m = gpc.models.BayesianGPLVM(Y, input_dim=2, num_inducing=10)
print(m)
print("kernel", m.kernel_type, "approx", m.approx)
before = m.objective()
m.optimize(iters=100, messages=False)
print("objective", before, "→", m.objective(), "ELBO", m.elbo())
mu_x = m.latent_mean()
s_x = m.latent_variance()
Z = m.inducing_inputs()
print("q(X) mean", mu_x.shape, "var", s_x.shape, "Z", Z.shape)

## Latent mean ± inducing inputs

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
for c, name in enumerate(("non-flowing", "slugging", "homogeneous")):
    mask = cls == c
    ax.scatter(mu_x[mask, 0], mu_x[mask, 1], s=28, alpha=0.8, label=name)
Z = m.inducing_inputs()
ax.plot(Z[:, 0], Z[:, 1], "kx", ms=8, mew=1.5, label="inducing")
ax.set_xlabel("latent 0")
ax.set_ylabel("latent 1")
ax.set_title("Bayesian GPLVM — oil_100")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()